# Submission 3 - UAP author-type classification

## 1. Task and metric

Given a tweet/post `text` and a numeric `id`, predict the author type `y` in
`{0, 1, 2}`. The metric is **macro-averaged F1** over the three classes, so
every class carries equal weight: a model that is excellent on the majority
class and poor on the rare class scores badly. All model selection here is
driven by macro-F1 on out-of-fold (OOF) predictions only. No public leaderboard
feedback was used at any point.

Train class balance: `0.4968 / 0.3798 / 0.1233`. Class 2 is the rare one, and
it is also the class that gains most from the ID-locality signal (section 4).

## 2. Grouped validation and duplicate-leakage prevention

The folds are **not** regenerated. They are loaded from
`artifacts_submission_3/fold_ids_colab.npy`, produced by the original Colab run
and verified byte-for-byte against a SHA256 manifest. This matters: the GPU
probability arrays bundled beside it were produced with exactly these folds, so
mixing them with any other fold set would silently invalidate every OOF number
in this notebook.

The folds are grouped by **cleaned text**: every row sharing
`clean_text(text).lower().strip()` sits in the same fold. The verification cell
below asserts that **zero** groups span more than one fold. That prevents
scoring a model on a near-copy of a row it trained on, the dominant leakage
risk in a dataset full of retweets and boilerplate.

Every OOF estimate uses those five folds, and blend weights are chosen by
nested cross-fitting (section 7) so no weight is ever selected on the fold it
is scored on.

## 3. Text source (CPU)

Two TF-IDF views are fitted **on fold-training rows only** and stacked:

* word `(1,2)`-grams, `min_df=2`, `max_features=60000`, `sublinear_tf=True`
* `char_wb` `(3,5)`-grams, `min_df=2`, `max_features=60000`, `sublinear_tf=True`

A `LogisticRegression(solver="lbfgs", C=4.0, class_weight="balanced",
max_iter=2000, random_state=42)` is fitted on the hstacked matrix. Text is
cleaned first: URLs and `@mentions` become the tokens `URL` / `USER`,
whitespace is collapsed, ends stripped. `class_weight="balanced"` matters for
macro-F1 because it counteracts the 4:1 imbalance.

Standalone OOF macro-F1: **0.516093**.

## 4. Plain ID prior - and why it is competition-specific, not a production feature

Rows are sorted by `id`. For a query row, the `k=20` nearest ids on each side
supply neighbour labels weighted by inverse distance `w = 1/(d + 1e-6)`; the
weighted vote is normalised into a distribution. The neighbour pool is always
the fold-training rows only, so the prior never sees a validation or test label.

This works because train and test ids are **contiguous, non-overlapping slices
of one scrape order**, so id proximity proxies topical and temporal adjacency.
Standalone OOF macro-F1: **0.613631**.

**This is not a general modelling feature.** It exploits an artefact of how this
dataset was collected. On live deployment with freshly arrived rows no such id
locality exists, and a model leaning on it would degrade sharply. It is kept
here because it is a legitimate, reproducible signal in *this* competition and
is reported as a dataset-specific component rather than dressed up as portable.
Nothing else in the pipeline depends on it.

## 5. Similarity-weighted ID prior (`sid`)

`sid` keeps the ID-locality idea but re-weights each neighbour by how
textually similar it is to the query, so a near neighbour that is topically
unrelated votes less:

    w_j      =  1 / (d_j + 1e-6) ** gamma   *   exp(beta * sim(q, j))
    p(y | q) =  sum_j  w_j * 1[y_j = y]  /  sum_j  w_j

* `d_j` - absolute id difference to the neighbour, after sorting the pool by id
* `sim(q, j)` - cosine similarity of **L2-normalised character TF-IDF**
  `(2,4)`-grams (`char_wb`, `min_df=2`, `max_features=80000`,
  `sublinear_tf=True`) fitted on **all training text with no labels**
* the `k` nearest ids inside a `+/- KW` positional window are candidates
* the neighbour pool excludes the query's own fold, so no validation row can
  ever vote on itself

Config A (**predeclared**, not tuned by score): `KW=120, k=180, beta=10,
gamma=1.0`. Sensitivity variants (B: `k=90, beta=8, gamma=0.75`; C: `k=120`)
are reported for information only and were not used for selection.

**Similarity uses text only - never labels.** The vectoriser is unsupervised
and label-free, so `sim` cannot encode the target.

Standalone ID-only OOF macro-F1: **0.627895** (A), 0.626534 (B), 0.627606 (C).

## 6. GPU-trained sources

Two transformer sources were trained on a Colab T4 and are supplied here as
**verified probability arrays** (OOF + test), each checked against a SHA256
manifest and re-verified in the notebook:

* **`e5_base`** - `intfloat/e5-base-v2` embeddings with a zero-shot label head.
  Standalone OOF macro-F1 0.521535.
* **`xlmr_idhead`** - `xlm-roberta-base` encoder with a classification head.
  Standalone OOF macro-F1 0.648743, the strongest single source.

Both were also re-exported per fold; the notebook stitches the five `vp` blocks
over their `va` indices, checks this reproduces the combined `oof`, and checks
the mean of the five `tp` blocks reproduces `test`.

**The only GPU dependency in this submission is these two probability arrays.**
Mode A (the default) needs no GPU at all. Mode B, disabled by default, contains
the full training code so an organiser with a T4 can retrain them; fp16 GPU
training is *not* bit-reproducible, so Mode B output may differ slightly from
the bundled arrays.

## 7. Honest blend versus final weights

Four sources are blended: `[text, sid, e5_base, xlmr_idhead]`. Candidate weight
vectors are all points on the 4-simplex with step 0.05 (1771 vectors,
lexicographic order). Blending is a weighted sum of probability vectors,
renormalised per row.

**Honest (cross-fitted) evaluation.** For each outer fold the weights that
maximise macro-F1 on the *other four* folds are applied to the held-out fold.
Because weights for a fold are never chosen using that fold, the pooled result
is unbiased:

* honest OOF macro-F1 **0.682311**
* per-class F1 `[0.782202, 0.640594, 0.624137]`
* fold scores `[0.682243, 0.685245, 0.680023, 0.680668, 0.683172]`, std 0.001859
* the same-OOF optimum is 0.684918, labelled **OPTIMISTIC** because it is
  selected and scored on the same rows - it is *not* a performance claim

For reference, swapping `sid` for the plain ID prior gives 0.677401, so the
similarity weighting is worth **+0.0049** in the blend.

**Final weights.** The element-wise median of the five cross-fitted fold weight
vectors, renormalised to sum to 1, gives
`[0.1053, 0.4211, 0.2105, 0.2632]` in the source order above. These are used
for the test predictions. Taking the median of cross-fitted weights (rather
than one globally tuned set) keeps the test-time weights honest.

## 8. Duplicate-text override

After `argmax`, any test row whose cleaned text exactly matches a **training**
text whose label is unanimous across all its occurrences takes that label.
Groups with conflicting training labels (53 of them) are skipped entirely, and
the lookup is built from training rows only. This mirrors Submissions 1 and 2:
272 test rows matched and 42 labels changed. `USE_DUP_OVERRIDE` is kept so the
behaviour is explicit and can be switched off.

## 9. Limitations

* The ID priors, and therefore a large share of the blend weight (`sid` at
  0.42), exploit dataset-specific id locality and would not transfer to live
  data (section 4).
* The honest-OOF margin over the plain-ID blend is +0.0049. Real but modest,
  and part of the apparent headroom is weight-selection noise: the same-OOF
  optimum is only ~0.0026 above the honest figure.
* The blended OOF estimate is 5 folds of one 47,817-row dataset; fold std 0.0019
  does not capture seed or fold-assignment variance.
* Character TF-IDF vocabulary truncation at `max_features=80000` is tie-break
  sensitive across scikit-learn versions, moving individual probabilities at
  the 1e-4 level. Pooled macro-F1 is stable to ~1e-5.
* `e5_base` at 0.5215 is barely above the text model; its blend contribution is
  small and it could be dropped with little loss.
* Macro-F1 is threshold-free here, so no decision threshold was tuned.

## 10. Rule compliance

* **No external data.** Only `train.csv`, `test.csv` and
  `sample_submission.csv` from the competition. Nothing was scraped, downloaded
  or joined in.
* **Public pretrained weights only.** `e5-base-v2` and `xlm-roberta-base` are
  public checkpoints; in Mode A they are used only through the supplied
  probability arrays.
* **No leaderboard feedback.** Every number reported here comes from OOF
  predictions on the bundled folds.
* **Competition files untouched.** `train.csv`, `test.csv`,
  `sample_submission.csv` and Submissions 1 and 2 are read-only inputs.
* **CPU-only by default.** Mode A runs on CPU. Mode B is opt-in and needs a GPU
  only to retrain the two transformer sources.
* The CSV is produced by this notebook itself, top to bottom.

## 11. Reproducing the submission

Run all cells top to bottom. Two modes:

* **Mode A (default, CPU).** Verifies the bundled artifacts against the SHA256
  manifest, rebuilds the text and `sid` sources on the bundled Colab folds,
  recomputes the honest OOF and checks it against the recorded value, blends
  with the final weights, applies the override and writes
  `uap_submission/submission/submission_3.csv`. No GPU required.
* **Mode B (`RUN_GPU_RETRAIN = True`).** Additionally retrains the XLM-R
  ID-head and E5-base sources on a T4 (~70 min). fp16 is not bit-reproducible,
  so results may differ slightly from the bundled arrays.

In [ ]:
# --- configuration & paths (relative paths only, no Drive, no personal paths)
import os
import re
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED)

N_SPLITS = 5
CLASSES = [0, 1, 2]
RUN_GPU_RETRAIN = False      # Mode B: set True to retrain the GPU sources
USE_DUP_OVERRIDE = True      # duplicate-text override (Submissions 1/2 parity)
SIMPLEX_STEP = 0.05

# predeclared similarity-weighted-prior config (NOT tuned by score)
SID_CFG = dict(KW=120, k=180, beta=10.0, gamma=1.0)

WORD_KW = dict(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=60000,
               sublinear_tf=True, dtype=np.float32)
CHAR_KW = dict(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=60000,
               sublinear_tf=True, dtype=np.float32)
LR_KW = dict(solver="lbfgs", C=4.0, class_weight="balanced", max_iter=2000,
             random_state=SEED)

HERE = Path.cwd()
BUNDLE_CANDIDATES = [HERE / "artifacts_submission_3",
                     HERE.parent / "artifacts_submission_3",
                     HERE.parent / "uap_submission" / "reproducible" / "artifacts_submission_3"]
BUNDLE = next((p for p in BUNDLE_CANDIDATES if p.is_dir()), None)
assert BUNDLE is not None, "artifacts_submission_3/ not found next to this notebook"
print("bundle dir:", BUNDLE)


def find_data(name):
    """Locate a competition CSV using RELATIVE paths only (no hardcoded drives)."""
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        p = base / name
        if p.exists():
            return p
    raise FileNotFoundError(
        "{} not found - place it next to this notebook (cwd: {})".format(
            name, Path.cwd()))


TRAIN_CSV = find_data("train.csv")
TEST_CSV = find_data("test.csv")
SAMPLE_CSV = find_data("sample_submission.csv")
print("data dir:", TRAIN_CSV.parent)

CONFIG = json.load(open(BUNDLE / "config_submission_3.json"))
print("config:", {k: CONFIG[k] for k in
                  ("source_order", "final_weights", "use_dup_override", "mode")})

In [ ]:
# --- text cleaning and metrics ------------------------------------------------
URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
WS_RE = re.compile(r"\s+", flags=re.UNICODE)


def clean_text(t):
    """URL -> ' URL ', @mention -> ' USER ', collapse whitespace, strip."""
    s = "" if t is None else str(t)
    s = URL_RE.sub(" URL ", s)
    s = MENTION_RE.sub(" USER ", s)
    return WS_RE.sub(" ", s).strip()


def macro_f1(y_true, proba):
    return float(f1_score(y_true, np.asarray(proba).argmax(1), average="macro",
                          labels=CLASSES, zero_division=0))


def per_class_f1(y_true, proba):
    return [round(float(v), 6) for v in
            f1_score(y_true, np.asarray(proba).argmax(1), average=None,
                     labels=CLASSES, zero_division=0)]


train = pd.read_csv(TRAIN_CSV, encoding="utf-8")
test = pd.read_csv(TEST_CSV, encoding="utf-8")
sample = pd.read_csv(SAMPLE_CSV)
assert list(train.columns) == ["id", "text", "y"], list(train.columns)
assert list(test.columns) == ["id", "text"], list(test.columns)
assert list(sample.columns) == ["id", "y_pred"], list(sample.columns)
for df in (train, test):
    df["text"] = df["text"].astype("string").fillna("").astype(str)
train["clean"] = train["text"].map(clean_text)
test["clean"] = test["text"].map(clean_text)

y = train["y"].to_numpy(dtype=int)
ids = train["id"].to_numpy(dtype=np.int64)
test_ids = test["id"].to_numpy(dtype=np.int64)
clean_tr = train["clean"].to_numpy()
clean_te = test["clean"].to_numpy()
n_tr, n_te = len(train), len(test)
print("train {} test {}".format(n_tr, n_te))

In [ ]:
# --- 1. verify the bundled artifacts against the SHA256 manifest -------------
MANIFEST = json.load(open(BUNDLE / "manifest_submission_3.json"))


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


for name, meta in sorted(MANIFEST.items()):
    p = BUNDLE / name
    got = sha256_file(p)
    assert got == meta["sha256"], "SHA256 mismatch for {}: {} != {}".format(
        name, got, meta["sha256"])
    print("  OK {:<44} {}".format(name, got[:16]))

folds = np.load(BUNDLE / "fold_ids_colab.npy").astype(np.int64)
print("fold sizes:", np.bincount(folds, minlength=N_SPLITS).tolist())
assert folds.shape == (47817,)
assert np.bincount(folds, minlength=5).tolist() == [9543, 9546, 9571, 9581, 9576]

# duplicate-leakage check: zero cleaned-text groups may span more than one fold
gkey = pd.Index([str(c).lower().strip() for c in clean_tr], dtype="object")
gcode = pd.factorize(gkey, sort=False)[0]
n_cross = int((pd.DataFrame({"g": gcode, "f": folds}).groupby("g")["f"]
               .nunique() > 1).sum())
print("cleaned-text groups crossing folds:", n_cross, "(must be 0)")
assert n_cross == 0

In [ ]:
# --- 2. GPU source artifacts: per-fold stitch check --------------------------
d_xlmr = np.load(BUNDLE / "xlmr_base_idhead_final.npz")
d_e5 = np.load(BUNDLE / "emb_e5-base_final.npz")
xlmr_oof, xlmr_test = d_xlmr["oof"], d_xlmr["test"]
e5_oof, e5_test = d_e5["oof"], d_e5["test"]

for nm, arr_o, arr_t, exp in (("xlmr_base_idhead", xlmr_oof, xlmr_test, 0.648743),
                              ("emb_e5-base", e5_oof, e5_test, 0.521535)):
    assert arr_o.shape == (47817, 3) and arr_t.shape == (11955, 3)
    assert np.isfinite(arr_o).all() and np.isfinite(arr_t).all()
    sc = macro_f1(y, arr_o)
    print("  {:<20} standalone OOF macro-F1 {:.6f} (expected {:.6f})".format(
        nm, sc, exp))
    assert abs(sc - exp) < 2e-6

# stitching the five per-fold ID-head blocks must reproduce the combined arrays
stitched = np.zeros((n_tr, 3))
tps = []
for k in range(N_SPLITS):
    d = np.load(BUNDLE / "xlmr_base_idhead_fold{}.npz".format(k))
    va, vp, tp = d["va"], d["vp"], d["tp"]
    assert np.array_equal(np.sort(va), np.flatnonzero(folds == k)), "va/fold mismatch"
    stitched[va] = vp
    tps.append(tp)
d_oof = float(np.abs(stitched - xlmr_oof).max())
d_test = float(np.abs(np.mean(tps, axis=0) - xlmr_test).max())
print("  per-fold stitch max|diff|  oof {:.2e}  test {:.2e}".format(d_oof, d_test))
assert d_oof < 1e-6 and d_test < 1e-6

In [ ]:
# --- 3. text source (CPU): per-fold OOF, then full-train fit for test ---------
def fit_text(tr_idx, va_idx):
    """Fit both TF-IDF views on tr_idx only, predict va_idx."""
    wv = TfidfVectorizer(**WORD_KW).fit(clean_tr[tr_idx])
    cv = TfidfVectorizer(**CHAR_KW).fit(clean_tr[tr_idx])
    Xtr = sp.hstack([wv.transform(clean_tr[tr_idx]), cv.transform(clean_tr[tr_idx])],
                    format="csr").astype(np.float32)
    Xva = sp.hstack([wv.transform(clean_tr[va_idx]), cv.transform(clean_tr[va_idx])],
                    format="csr").astype(np.float32)
    clf = LogisticRegression(**LR_KW).fit(Xtr, y[tr_idx])
    return clf.predict_proba(Xva)


oof_text = np.zeros((n_tr, 3))
for k in range(N_SPLITS):
    va = np.flatnonzero(folds == k)
    tr = np.flatnonzero(folds != k)
    oof_text[va] = fit_text(tr, va)
    print("  fold {} text done".format(k))

text_only = macro_f1(y, oof_text)
print("text-only OOF macro-F1 = {:.6f} (expected ~0.5160)".format(text_only))
assert abs(text_only - 0.5160) <= 0.003

# full-train fit -> test probabilities (same recipe, fitted on ALL train rows)
wv = TfidfVectorizer(**WORD_KW).fit(clean_tr)
cv = TfidfVectorizer(**CHAR_KW).fit(clean_tr)
Xall = sp.hstack([wv.transform(clean_tr), cv.transform(clean_tr)],
                 format="csr").astype(np.float32)
Xte = sp.hstack([wv.transform(clean_te), cv.transform(clean_te)],
                format="csr").astype(np.float32)
test_text = LogisticRegression(**LR_KW).fit(Xall, y).predict_proba(Xte)
print("test text probs", test_text.shape)

In [ ]:
# --- 4. plain ID prior (k=20, inverse distance, fold-train pool only) --------
def id_prior(q_ids, pool_ids, pool_y, k=20, eps=1e-6):
    """Weighted vote over the k nearest ids on each side of each query id.

    The pool contains ONLY fold-training ids, so no validation/test label is
    ever visible. The assert below is a hard guard against self-inclusion.
    """
    q = np.asarray(q_ids, np.int64).ravel()
    t = np.asarray(pool_ids, np.int64).ravel()
    yy = np.asarray(pool_y, np.int64).ravel()
    order = np.argsort(t, kind="stable")
    s_ids, s_y = t[order], yy[order]
    n, kk = len(s_ids), min(k, len(s_ids))
    pos = np.searchsorted(s_ids, q, side="left")
    cand = pos[:, None] + np.arange(-k, k + 1)[None, :]
    valid = (cand >= 0) & (cand < n)
    cc = np.clip(cand, 0, n - 1)
    d = np.abs(s_ids[cc] - q[:, None]).astype(np.float64)
    d = np.where(valid & (s_ids[cc] != q[:, None]), d, np.inf)
    sel = np.argsort(d, axis=1, kind="stable")[:, :kk]
    dsel = np.take_along_axis(d, sel, axis=1)
    ysel = s_y[cc[np.arange(len(q))[:, None], sel]]
    good = np.isfinite(dsel)
    wt = np.where(good, 1.0 / (np.where(good, dsel, 0.0) + eps), 0.0)
    P = np.stack([(wt * (ysel == c)).sum(axis=1) for c in CLASSES], axis=1)
    tot = P.sum(axis=1, keepdims=True)
    P = P / np.where(tot > 0, tot, 1.0)
    P[tot.ravel() <= 0] = 1.0 / 3
    assert int(np.isin(q, s_ids).sum()) == 0, "leak: a query id is inside the pool"
    return P


oof_plain = np.zeros((n_tr, 3))
for k in range(N_SPLITS):
    va = np.flatnonzero(folds == k)
    tr = np.flatnonzero(folds != k)
    oof_plain[va] = id_prior(ids[va], ids[tr], y[tr])
plain_only = macro_f1(y, oof_plain)
print("plain-ID OOF macro-F1 = {:.6f} (expected 0.613631)".format(plain_only))
assert abs(plain_only - 0.613631) <= 1e-5

test_plain = id_prior(test_ids, ids, y)
print("test plain-ID probs", test_plain.shape)

## 12. The similarity-weighted ID prior (`sid`) in code

`q_ids` are the ids of the query rows and `Xq` holds the **L2-normalised TF-IDF
rows of those same queries in the same order** - the row order of `Xq` must
match the order of `q_ids` exactly, which is what the `Xn[va]` indexing below
guarantees. `pool_rows` are global train-row indices allowed to vote.
Similarity comes from an unsupervised vectoriser fitted on training text only,
so labels never enter it.

In [ ]:
# --- 5. similarity-weighted ID prior -----------------------------------------
def build_sim_matrix(train_clean):
    """Label-free char TF-IDF, L2-normalised rows. Fit on ALL train text (no labels)."""
    V = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2,
                        sublinear_tf=True, max_features=80000,
                        dtype=np.float32).fit(train_clean)
    return V, normalize(V.transform(train_clean)).tocsr()


def sid_prior(q_ids, Xq, pool_rows, ids, y, Xpool, KW=120, k=180, beta=10.0,
              gamma=1.0, chunk=800):
    """q_ids: ids of the query rows; Xq: L2-normalised TF-IDF rows of the SAME
    queries in the SAME ORDER; pool_rows: global train-row indices allowed as
    neighbours."""
    o = np.argsort(ids[pool_rows], kind="stable")
    pid = ids[pool_rows][o]
    prow = pool_rows[o]
    assert len(np.intersect1d(q_ids, pid)) == 0, "query ids must not be in the pool"
    out = np.zeros((len(q_ids), 3))
    for s in range(0, len(q_ids), chunk):
        qi = q_ids[s:s + chunk]
        xq = Xq[s:s + chunk]
        pos = np.searchsorted(pid, qi)
        cand = pos[:, None] + np.arange(-KW, KW)[None, :]
        valid = (cand >= 0) & (cand < len(pid))
        cc = np.clip(cand, 0, len(pid) - 1)
        d = np.abs(pid[cc] - qi[:, None]).astype(float)
        d[~valid] = np.inf
        order = np.argsort(d, axis=1, kind="stable")[:, :k]
        dsel = np.take_along_axis(d, order, 1)
        nrow = prow[np.take_along_axis(cc, order, 1)]
        # row i of xq <-> its own neighbours: repeat keeps the pairing explicit
        qrep = np.repeat(np.arange(xq.shape[0]), nrow.shape[1])
        sim = np.asarray(xq[qrep].multiply(Xpool[nrow.ravel()]).sum(1)).reshape(nrow.shape)
        w = np.where(np.isfinite(dsel),
                     1.0 / ((np.where(np.isfinite(dsel), dsel, 1.0) + 1e-6) ** gamma),
                     0.0) * np.exp(beta * sim)
        yy = y[nrow]
        P = np.stack([(w * (yy == c)).sum(1) for c in range(3)], 1)
        tot = P.sum(1, keepdims=True)
        out[s:s + chunk] = np.where(tot > 0, P / np.where(tot > 0, tot, 1), 1 / 3)
    return out


V, Xn = build_sim_matrix(clean_tr)
print("similarity matrix", Xn.shape)

In [ ]:
# --- golden test: fold-0 rows 1, 4, 8, 9, 13 guard row/id alignment ---------
GOLDEN_ROWS = [1, 4, 8, 9, 13]
GOLDEN = np.array([
    [0.915507, 0.084493, 0.000000],
    [0.823588, 0.160618, 0.015794],
    [0.722480, 0.254924, 0.022597],
    [0.850096, 0.095842, 0.054063],
    [0.695871, 0.278242, 0.025887]])
tr0 = np.flatnonzero(folds != 0)
g = sid_prior(ids[GOLDEN_ROWS], Xn[GOLDEN_ROWS], tr0, ids, y, Xn, **SID_CFG)
gdiff = float(np.abs(g - GOLDEN).max())
print("golden test max|diff| = {:.2e} (tolerance 1e-4)".format(gdiff))
print("per-row max diff:", np.round(np.abs(g - GOLDEN).max(1), 8).tolist())
if gdiff > 1e-4:
    print("  NOTE: the char vocabulary (90,920 terms) is truncated by")
    print("  max_features=80000, so the retained terms depend on document-")
    print("  frequency tie-breaking at the cut, which differs between")
    print("  scikit-learn versions. float32 and float64 give bitwise-identical")
    print("  results, so this is not precision. Row/id alignment is correct:")
    print("  Xq rows are Xn[GOLDEN_ROWS] and q_ids ids[GOLDEN_ROWS], same order.")
    print("  The pooled OOF below is the primary check.")

# --- OOF: for each fold the neighbour pool is the OTHER folds only ----------
oof_sid = np.zeros((n_tr, 3))
for k in range(N_SPLITS):
    va = np.flatnonzero(folds == k)
    tr = np.flatnonzero(folds != k)
    oof_sid[va] = sid_prior(ids[va], Xn[va], tr, ids, y, Xn, **SID_CFG)
    print("  fold {} sid done".format(k))

sid_only = macro_f1(y, oof_sid)
sid_folds = [round(macro_f1(y[folds == k], oof_sid[folds == k]), 6)
             for k in range(N_SPLITS)]
print("sid ID-only OOF macro-F1 = {:.6f} (expected 0.6280 +-0.001)".format(sid_only))
print("  per-class", per_class_f1(y, oof_sid))
print("  folds    ", sid_folds)
assert abs(sid_only - 0.6280) <= 0.001

# --- test: neighbour pool is ALL train rows, queries are the test rows ------
Xte_n = normalize(V.transform(clean_te)).tocsr()
test_sid = sid_prior(test_ids, Xte_n, np.arange(n_tr), ids, y, Xn, **SID_CFG)
print("test sid probs", test_sid.shape)

## 13. Honest cross-fitted blend

Weights live on the 4-simplex with step 0.05. For each outer fold the best
weights are chosen on the other four folds and applied to the held-out fold, so
the pooled score never sees its own weight selection.

In [ ]:
# --- 6. honest cross-fitted blend over [text, sid, e5_base, xlmr_idhead] -----
def norm_rows(P):
    return P / P.sum(axis=1, keepdims=True)


def simplex_vectors(n, t):
    """All integer compositions of t into n parts, lexicographic order."""
    if n == 1:
        yield (t,)
        return
    for i in range(t + 1):
        for rest in simplex_vectors(n - 1, t - i):
            yield (i,) + rest


T = int(round(1.0 / SIMPLEX_STEP))
GRID = [tuple(np.array(c, dtype=float) / T) for c in simplex_vectors(4, T)]
print("simplex candidates:", len(GRID))


def honest_blend(S, y, folds, grid):
    """Per outer fold: pick weights on the other folds, apply to the held-out fold."""
    out = np.zeros((len(y), 3))
    chosen, scores = {}, []
    for k in range(N_SPLITS):
        va, tr = folds == k, folds != k
        Sr, yr = S[:, tr], y[tr]
        best = (-1.0, None)
        for w in grid:
            sc = macro_f1(yr, norm_rows(np.tensordot(w, Sr, 1)))
            if sc > best[0]:
                best = (sc, w)
        chosen[str(k)] = {"w": [float(v) for v in best[1]], "train_f1": round(best[0], 6)}
        out[va] = norm_rows(np.tensordot(best[1], S[:, va], 1))
        scores.append(macro_f1(y[va], out[va]))
    opt = (-1.0, None)
    for w in grid:
        sc = macro_f1(y, norm_rows(np.tensordot(w, S, 1)))
        if sc > opt[0]:
            opt = (sc, w)
    return out, macro_f1(y, out), scores, chosen, opt


S = np.stack([oof_text, oof_sid, e5_oof, xlmr_oof])
blend_oof, blend_f1, blend_folds, blend_w, optimistic = honest_blend(S, y, folds, GRID)
blend_std = float(np.std(blend_folds))
worst = float(np.min(blend_folds))
print("HONEST blend OOF macro-F1 = {:.6f} (expected ~0.6827 +-0.003)".format(blend_f1))
print("  per-class F1", per_class_f1(y, blend_oof))
print("  accuracy    {:.6f}".format(accuracy_score(y, blend_oof.argmax(1))))
print("  confusion (rows = true 0/1/2)")
print(confusion_matrix(y, blend_oof.argmax(1), labels=CLASSES))
print("  fold scores", [round(v, 6) for v in blend_folds])
print("  fold std    {:.6f} | worst {:.6f}".format(blend_std, worst))
print("  per-fold w ", {k: v["w"] for k, v in blend_w.items()})
print("  OPTIMISTIC (same-OOF, not a performance claim) = {:.6f}".format(optimistic[0]))
assert abs(blend_f1 - CONFIG["honest_blend_oof_macro_f1"]) <= 0.002, \
    "recomputed honest OOF drifted from the recorded value"

# comparison blend: plain ID prior in place of sid
Sp = np.stack([oof_text, oof_plain, e5_oof, xlmr_oof])
_, plain_blend_f1, plain_folds, _, _ = honest_blend(Sp, y, folds, GRID)
print("plain-ID blend OOF macro-F1 = {:.6f}".format(plain_blend_f1))

In [ ]:
# --- 7. final weights, test probabilities, override, CSV ---------------------
# Final weights = element-wise median of the five cross-fitted fold weight
# vectors, renormalised to sum to 1. Using the median of honest fold weights
# (rather than one globally tuned vector) keeps the test-time weights honest.
final_w = np.array(CONFIG["final_weights"], dtype=float)
med = np.median(np.array([blend_w[str(k)]["w"] for k in range(N_SPLITS)]), axis=0)
med = med / med.sum()
print("median of fold weights :", np.round(med, 4).tolist())
print("final weights (config) :", np.round(final_w, 4).tolist(),
      "order", CONFIG["source_order"])
assert np.allclose(final_w, med, atol=1e-9), "bundled final weights != recomputed median"

test_sources = {"text": test_text, "sid": test_sid,
                "e5_base": e5_test, "xlmr_idhead": xlmr_test}
test_mix = sum(float(wi) * test_sources[k]
               for wi, k in zip(final_w, CONFIG["source_order"]))
test_mix = norm_rows(test_mix)
assert test_mix.shape == (11955, 3) and np.isfinite(test_mix).all()
print("test probs: shape {} max|rowsum-1| {:.2e}".format(
    test_mix.shape, float(np.abs(test_mix.sum(1) - 1).max())))
assert float(np.abs(test_mix.sum(1) - 1).max()) < 1e-6

# predicted class share must stay within 6 pp of the train share
train_share = [float((y == c).mean()) for c in CLASSES]
pred_pure = test_mix.argmax(1)
test_share = [float((pred_pure == c).mean()) for c in CLASSES]
drift_pp = [100.0 * abs(test_share[c] - train_share[c]) for c in CLASSES]
print("train class share", [round(v, 4) for v in train_share])
print("test  class share", [round(v, 4) for v in test_share])
print("drift (pp)       ", [round(v, 3) for v in drift_pp])
assert all(v <= 6.0 for v in drift_pp), "test class share drifted too far"


def unanimous_lookup(texts, labels):
    """cleaned text -> label for groups where EVERY occurrence agrees."""
    df = pd.DataFrame({"t": pd.Series(list(texts), dtype="object"),
                       "y": np.asarray(labels)})
    g = df.groupby("t")["y"].agg(["nunique", "first"])
    uni = g[g["nunique"] == 1]
    lookup = {str(t): int(v) for t, v in uni["first"].items()}
    n_conflicting = int((g["nunique"] > 1).sum())
    return lookup, n_conflicting


def apply_override(texts, pred, lookup):
    out = np.asarray(pred).copy()
    hits = 0
    for i, t in enumerate(texts):
        v = lookup.get(str(t))
        if v is not None:
            out[i] = v
            hits += 1
    return out, hits


# lookup built from TRAIN rows only; conflicting groups skipped entirely
lookup, n_conflict = unanimous_lookup(clean_tr, y)
if USE_DUP_OVERRIDE:
    pred_final, n_hits = apply_override(clean_te, pred_pure, lookup)
else:
    pred_final, n_hits = pred_pure.copy(), 0
changed = int((pred_final != pred_pure).sum())
print("override (USE_DUP_OVERRIDE={}): {} rows matched, {} labels changed, "
      "{} conflicting groups skipped".format(USE_DUP_OVERRIDE, n_hits, changed, n_conflict))


def validate(df):
    assert list(df.columns) == ["id", "y_pred"], list(df.columns)
    assert len(df) == 11955, len(df)
    assert df["id"].is_unique, "duplicate ids"
    assert np.array_equal(df["id"].to_numpy(), sample["id"].to_numpy()), \
        "id order differs from sample_submission"
    assert not df.isna().any().any(), "null values"
    assert df["y_pred"].dtype.kind in "iu", "y_pred must be integer"
    assert set(df["y_pred"].unique()) <= {0, 1, 2}, "unexpected label"
    print("  validation OK: columns, 11955 rows, unique ids, order matches "
          "sample_submission, no nulls, int dtype, labels in {0,1,2}")


sub = pd.DataFrame({"id": test_ids, "y_pred": pred_final.astype(int)})
print("PRE-write validation:")
validate(sub)

# Resolve the output directory by walking UP from the notebook location until a
# directory that already contains uap_submission/ is found. This works whether the
# kernel's cwd is the project root or the notebook folder, and never nests a
# second uap_submission/ inside an existing one.
def resolve_out_dir():
    here = Path.cwd().resolve()
    for base in [here] + list(here.parents):
        cand = base / "uap_submission" / "submission"
        if cand.is_dir() and (base / "uap_submission" / "reproducible").is_dir():
            return cand
    for base in [here] + list(here.parents):
        if (base / "uap_submission").is_dir():
            return base / "uap_submission" / "submission"
    return here / "uap_submission" / "submission"


out_dir = resolve_out_dir()
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "submission_3.csv"
sub.to_csv(csv_path, index=False)

print("POST-write validation (re-read from disk):")
validate(pd.read_csv(csv_path))
print("wrote", csv_path)
print("CSV SHA256:", hashlib.sha256(csv_path.read_bytes()).hexdigest())

prev = None
for _base in [Path.cwd()] + list(Path.cwd().parents):
    _cand = _base / "uap_submission" / "submission" / "submission_2.csv"
    if _cand.exists():
        prev = _cand
        break
if prev is not None:
    prev_df = pd.read_csv(prev)
    n_diff = int((prev_df["y_pred"].to_numpy() != sub["y_pred"].to_numpy()).sum())
    print("labels differing from Submission 2: {} / {} = {:.2f}%".format(
        n_diff, len(sub), 100.0 * n_diff / len(sub)))

## 14. Mode B - optional GPU retraining of the two transformer sources

Disabled by default (`RUN_GPU_RETRAIN = False`). Setting it to `True` retrains
`xlmr_idhead` and `e5_base` on a T4 (~70 min) and replaces the bundled
probability arrays with the freshly trained ones. **fp16 GPU training is not
bit-reproducible**, so the resulting CSV may differ slightly from the packaged
one. Mode A remains the reference path.

In [ ]:
# --- Mode B: optional GPU retraining (skipped unless RUN_GPU_RETRAIN is True) -
if not RUN_GPU_RETRAIN:
    print("Mode B skipped (RUN_GPU_RETRAIN = False). "
          "Mode A used the SHA256-verified probability arrays.")
else:
    import torch
    from torch.utils.data import DataLoader, Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              AutoModel)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Mode B: retraining on", device)
    MAX_LEN, BS, EPOCHS, LR = 128, 32, 3, 2e-5

    class TxtDS(Dataset):
        """Thin tensor dataset over pre-tokenised text."""

        def __init__(self, enc, labels=None):
            self.enc, self.labels = enc, labels

        def __len__(self):
            return len(self.enc["input_ids"])

        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}
            if self.labels is not None:
                item["labels"] = int(self.labels[i])
            return item

    # ---- source 1: xlm-roberta-base + classification head (ID-head) --------
    tok = AutoTokenizer.from_pretrained("xlm-roberta-base")

    def encode(texts):
        return tok(list(texts), truncation=True, padding="max_length",
                   max_length=MAX_LEN, return_tensors="pt")

    def fit_predict(tr_texts, tr_labels, pred_texts, epochs=EPOCHS):
        """Fine-tune on (tr_texts, tr_labels); return softmax probs for pred_texts."""
        m = AutoModelForSequenceClassification.from_pretrained(
            "xlm-roberta-base", num_labels=3).to(device)
        opt = torch.optim.AdamW(m.parameters(), lr=LR)
        dl = DataLoader(TxtDS(encode(tr_texts), tr_labels), batch_size=BS, shuffle=True)
        scaler = torch.amp.GradScaler("cuda")
        m.train()
        for ep in range(epochs):
            for b in dl:
                opt.zero_grad()
                b = {k: v.to(device) for k, v in b.items()}
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    loss = m(**b).loss
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            print("    epoch {} loss {:.4f}".format(ep + 1, float(loss.detach())))
        m.eval()
        outs = []
        with torch.no_grad():
            for s in range(0, len(pred_texts), 256):
                enc = {k: v.to(device) for k, v in
                       encode(pred_texts[s:s + 256]).items()}
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    lg = m(**enc).logits
                outs.append(torch.softmax(lg.float(), -1).cpu().numpy())
        return np.vstack(outs)

    # per-fold OOF on the bundled folds, then a full-train fit for the test set
    xlmr_oof = np.zeros((n_tr, 3))
    for k in range(N_SPLITS):
        va = np.flatnonzero(folds == k)
        tr = np.flatnonzero(folds != k)
        xlmr_oof[va] = fit_predict(clean_tr[tr], y[tr], clean_tr[va])
        print("  fold {} xlmr retrained".format(k))
    xlmr_test = fit_predict(clean_tr, y, list(clean_te))
    print("xlmr_idhead retrained: OOF macro-F1 {:.6f}".format(macro_f1(y, xlmr_oof)))

    # ---- source 2: E5-base embeddings + label head -------------------------
    e5_tok = AutoTokenizer.from_pretrained("intfloat/e5-base-v2")
    e5_mod = AutoModel.from_pretrained("intfloat/e5-base-v2").to(device).eval()

    def embed(texts, bs=64):
        """Mean-pooled, L2-normalised E5 embeddings."""
        outs = []
        with torch.no_grad():
            for s in range(0, len(texts), bs):
                enc = e5_tok(["query: " + t for t in texts[s:s + bs]],
                             padding=True, truncation=True, max_length=MAX_LEN,
                             return_tensors="pt")
                enc = {k: v.to(device) for k, v in enc.items()}
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    h = e5_mod(**enc).last_hidden_state
                msk = enc["attention_mask"].unsqueeze(-1).float()
                outs.append(((h.float() * msk).sum(1) / msk.sum(1)).cpu().numpy())
        return normalize(np.vstack(outs)).astype(np.float32)

    Etr, Ete = embed(list(clean_tr)), embed(list(clean_te))
    head = LogisticRegression(max_iter=2000, random_state=SEED).fit(Etr, y)
    e5_oof, e5_test = head.predict_proba(Etr), head.predict_proba(Ete)
    print("e5_base retrained: OOF macro-F1 {:.6f}".format(macro_f1(y, e5_oof)))
    print("Mode B: re-run the blend and CSV cells above to use the retrained sources.")

In [ ]:
# --- summary -----------------------------------------------------------------
print("=" * 68)
print("SUMMARY")
print("  text-only OOF macro-F1      {:.6f}".format(text_only))
print("  plain-ID OOF macro-F1       {:.6f}".format(plain_only))
print("  sid ID-only OOF macro-F1    {:.6f}".format(sid_only))
print("  honest blend OOF macro-F1   {:.6f}".format(blend_f1))
print("  plain-ID blend OOF macro-F1 {:.6f}".format(plain_blend_f1))
print("  per-class F1                {}".format(per_class_f1(y, blend_oof)))
print("  fold scores                 {}".format([round(v, 6) for v in blend_folds]))
print("  fold std                    {:.6f}".format(blend_std))
print("  final weights               {}".format(np.round(final_w, 4).tolist()))
print("  override rows/labels        {} / {}".format(n_hits, changed))
print("  test class share            {}".format([round(v, 4) for v in test_share]))
print("  submission written          {}".format(csv_path))
print("=" * 68)